1. Install Dependencies

In [ ]:
! pip install transformers torch # Install the transformers and torch libraries for natural language processing and deep learning tasks

2. Import Libraries

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer # Import the AutoModelForCausalLM and AutoTokenizer classes from the transformers library for loading pre-trained language models and tokenizers
import torch # Import the torch library for tensor operations and deep learning functionalities

3. Load Model

In [ ]:
model_name = "microsoft/DialoGPT-medium" # Specify the name of the pre-trained language model to be used for the chatbot (in this case, Microsoft DialoGPT-medium)

tokenizer = AutoTokenizer.from_pretrained(model_name) # Load the tokenizer associated with the specified pre-trained language model to convert text into token IDs that can be processed by the model
model = AutoModelForCausalLM.from_pretrained(model_name) # Load the pre-trained language model specified by model_name, which will be used to generate responses in the chatbot based on the input text and conversation history

pytorch_model.bin:   0%|          | 0.00/863M [00:00<?, ?B/s]

d:\Anaconda\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jaysh\.cache\huggingface\hub\models--microsoft--DialoGPT-medium. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/863M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

4. Chatbot Logic

In [ ]:
chat_history_ids = None # Initialize chat_history_ids to None, which will be used to store the conversation history in the form of token IDs for generating context-aware responses from the model

print("Chatbot: Hello! I am your AI assistant. Type 'exit' to end.\n") # Print a welcome message to the user, indicating that they can start interacting with the chatbot and how to exit the conversation

def smart_response(user_input):
    user_input_lower = user_input.lower() # Convert the user input to lowercase to enable case-insensitive matching for predefined responses

    if "what is artificial intelligence" in user_input_lower or "what is ai" in user_input_lower: # Check if the user input contains a question about artificial intelligence (AI) by looking for specific keywords in the lowercase version of the input
        return "Artificial Intelligence refers to the simulation of human intelligence by machines that can perform tasks such as learning, reasoning, and problem solving."

    elif "who created python" in user_input_lower: # Check if the user input contains a question about the creator of Python by looking for specific keywords in the lowercase version of the input
        return "Python was created by Guido van Rossum and released in 1991."

    elif "thank you" in user_input_lower: # Check if the user input contains a thank you message by looking for the phrase "thank you" in the lowercase version of the input
        return "You're welcome! Feel free to ask more questions."

    return None  # fallback to model # If the user input does not match any of the predefined responses, return None to indicate that the chatbot should generate a response using the language model based on the conversation history and user input


while True: # Start an infinite loop to continuously interact with the user until they choose to exit the conversation
    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit"]: # Check if the user input is a command to exit the conversation by looking for specific keywords (case-insensitive) in the user input
        print("Chatbot: Goodbye! Have a great day!")
        break

    custom_reply = smart_response(user_input) # Call the smart_response function with the user input to check for predefined responses based on specific keywords or phrases in the input. If a predefined response is found, it will be returned; otherwise, None will be returned to indicate that the chatbot should generate a response using the language model.

    if custom_reply:
        print("Chatbot:", custom_reply) # If a predefined response is found (i.e., custom_reply is not None), print the chatbot's response to the user and skip the language model generation step for this input
        continue

    new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt') # Encode the user input into token IDs using the tokenizer, appending the end-of-sequence token to indicate the end of the user's message. The return_tensors='pt' argument specifies that the output should be returned as a PyTorch tensor, which is required for input to the language model.

    bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1) if chat_history_ids is not None else new_input_ids # Concatenate the new input token IDs with the existing conversation history token IDs (if any) to create the complete input for the language model. If chat_history_ids is None (i.e., this is the first user input), then bot_input_ids will simply be the new_input_ids.

    attention_mask = torch.ones(bot_input_ids.shape, dtype=torch.long) # Create an attention mask of the same shape as bot_input_ids, filled with ones, to indicate that all tokens in the input should be attended to by the model during generation.

    chat_history_ids = model.generate( # Generate a response from the language model using the complete input (bot_input_ids) and the attention mask. The generation parameters specify the maximum length of the response, padding token ID, sampling strategy (top-k and top-p), and temperature for controlling randomness in the generated response.
        bot_input_ids,
        attention_mask=attention_mask,
        max_length=200,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_k=50,
        top_p=0.9,
        temperature=0.6
    )

    response = tokenizer.decode( # Decode the generated response from the model back into human-readable text using the tokenizer. The chat_history_ids tensor contains the complete conversation history, so we slice it to get only the newly generated response by taking the tokens starting from the length of bot_input_ids (which includes both the conversation history and the new user input). The skip_special_tokens=True argument ensures that special tokens (like end-of-sequence) are not included in the decoded response.
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    print("Chatbot:", response) # Print the chatbot's generated response to the user, allowing them to continue the conversation by providing new input in the next iteration of the loop

Chatbot: Hello! I am your AI assistant. Type 'exit' to end.

Chatbot: Hey there!
Chatbot: Artificial Intelligence refers to the simulation of human intelligence by machines that can perform tasks such as learning, reasoning, and problem solving.
Chatbot: Python was created by Guido van Rossum and released in 1991.
Chatbot: You're welcome! Feel free to ask more questions.
Chatbot: Goodbye! Have a great day!


5. How It Works

1. User input is converted into tokens
2. Model processes input using transformer architecture
3. Generates response using probability distribution
4. Maintains conversation using chat_history_ids